# Task 1 — The Messy Sales Log

**Ali Raza** · Roll No: **su92-bsdsm-f23-018** · Section: **BSDS-7A**
Data Warehouse & BI — Week 1

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#9aa4b2", "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937", "xtick.color": "#4b5563", "ytick.color": "#4b5563",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.7,
    "axes.axisbelow": True, "font.size": 10,
})
INK, BLUE, RED, GREY = "#1f2937", "#2563eb", "#dc2626", "#9ca3af"
os.makedirs("charts", exist_ok=True)
NOTE = "DWBI Week 1 · synthetic course data"


## The file

38 rows, one morning's orders from three branches — exactly as the branch systems exported it.
First, look at it raw:

In [2]:
raw = pd.read_csv("data/DWBI_W1_Messy_Sales_Log.csv", dtype=str)
print(raw.shape)
print(raw.head(10).to_string(index=False))

(38, 7)
OrderID         Branch       Date          Product   Category Qty Amount
   1001    LHR-Gulberg 12/09/2026        Cola 1.5L  Beverages   2    360
   1002 Lahore Gulberg 2026-09-12           CL-150       Bev.   1 Rs 180
   1003    KHI-Clifton 09-12-2026     Cola 1.5 Ltr  Beverages   3    540
   1004         ISB-F7  12 Sep 26 Masala Chips 50g     Snacks   4    200
   1004         ISB-F7  12 Sep 26 Masala Chips 50g     Snacks   4    200
   1005    khi clifton 12/09/2026    Fresh Milk 1L        NaN   2    440
   1006    LHR-Gulberg 2026-09-12 Basmati Rice 5kg    Grocery   1  2,450
   1007         ISB-F7 12/09/2026     LED Bulb 12W Electrical  -1   -350
   1008 Lahore Gulberg 12/09/2026        Cola 1.5L  Beverages   2   3600
   1009        ISB F-7 2026/09/12      Yogurt 500g      Dairy   3    510


## Seven problems — each with an OrderID, why it breaks a report, and who fixes it

| # | Problem (OrderID) | Why it breaks a report | Owner — why |
|---|---|---|---|
| 1 | **One branch, four spellings** — `1002` says `Lahore Gulberg`, `1001` says `LHR-Gulberg` (also `ISB-F7` vs `ISB F-7`, `KHI-Clifton` vs `khi clifton`) | `GROUP BY Branch` returns **six** branches instead of three; Lahore's sales split across spellings | **ETL** — the three real branches are known, so staging conforms every spelling to one canonical branch dimension, once, for every report |
| 2 | **Five date formats** — `1004` uses `12 Sep 26` while others use `12/09/2026`, `2026-09-12`, `09-12-2026`, `2026/09/12` | date parsing fails or, worse, `09-12-2026` reads as 9 Dec in one tool and 12 Sep in another | **ETL** — parse every variant to one ISO date on the way in; no report should ever see five formats |
| 3 | **Duplicated order** — `1004` appears twice, identical in every column | every total double-counts it; revenue is overstated by Rs 200 | **Source system** — a duplicate should never have been recorded; the POS needs a unique order key so a retry can't create two rows |
| 4 | **One product, three names** — `1002` is `CL-150`, `1001` is `Cola 1.5L`, `1003` is `Cola 1.5 Ltr` (same for `Orange Juice 1L` / `OJ-1L`, `Fresh Milk 1L` / `Milk 1 Litre`) | product-level reporting fragments: Cola's sales scatter across three "products" | **ETL** — map every alias to one canonical SKU in staging; the mapping table is versioned so history stays consistent |
| 5 | **Blank category** — `1005` (`Fresh Milk 1L`) and `1033` (`Milk 1 Litre`) have no category at all | they vanish from every category report — the CEO's Beverages total silently excludes them | **The business** — no pipeline can decide what milk *is*; a human must write the rule (is milk Beverages or Dairy?) and only then can ETL apply it |
| 6 | **Amounts stored as text** — `1002` is `Rs 180`, `1006` is `"2,450"` | a naive `SUM` either errors or **silently skips** these rows — the total is wrong with no warning | **ETL** — strip currency symbols and separators and store a real number; every downstream SUM then just works |
| 7 | **Return as a negative row** — `1007` has Qty −1, Amount −350 | net vs gross revenue depends on a convention nobody wrote down; one analyst nets it, another drops it | **The business** — the business must define whether "revenue" means gross sales or net of returns; ETL then implements that definition everywhere, identically |
| 8 | **Decimal-point error** — `1008` charges **3600** for 2× Cola 1.5L; `1001` sells the same 2× for **360** | one keystroke inflates the Beverages total by Rs 3,240 — ten times the real price | **Source system** — the POS should validate the amount against the price master (2 × 180 ≠ 3600); by the time it reaches ETL the truth is already lost |

*(Eight found; the task asked for at least six.)*

In [3]:
# Prove problem 8 with the unit price: every cola row charges 180/unit — except 1008
d = raw.copy()
d["amt"] = d.Amount.str.replace("Rs", "", regex=False).str.replace(",", "", regex=False).str.strip().astype(float)
d["qty"] = d.Qty.astype(int)
cola = d[d.Product.str.contains("Cola|CL-150", na=False)]
for _, r in cola.iterrows():
    print(f"{r.OrderID}  {r.Product:14s} qty={r.qty}  amt={r.amt:7.0f}  unit={r.amt/r.qty:6.0f}")

1001  Cola 1.5L      qty=2  amt=    360  unit=   180
1002  CL-150         qty=1  amt=    180  unit=   180
1003  Cola 1.5 Ltr   qty=3  amt=    540  unit=   180
1008  Cola 1.5L      qty=2  amt=   3600  unit=  1800
1021  CL-150         qty=2  amt=    360  unit=   180
1030  Cola 1.5L      qty=1  amt=    180  unit=   180
1035  Cola 1.5L      qty=1  amt=    180  unit=   180


## The Beverages total — with every decision stated

**Decisions:**
1. **Duplicate `1004`** — excluded one copy. Effect on Beverages total: **Rs 0** (it's Snacks), but the file-level total would be overstated by Rs 200.
2. **Blank category (`1005`, `1033` — milk)** — the business has written no rule, so the strict Beverages total **excludes** them; I report the with-milk figure separately. Effect: **+Rs 1,320** if milk is ruled a beverage.
3. **Return (`1007`)** — kept as recorded. Effect on Beverages total: **Rs 0** (it's an LED bulb, not a beverage).
4. **`1008` decimal error** — corrected 3600 → **360**, the price every other cola row proves (2 × 180). Effect: **−Rs 3,240** vs the naive sum.

In [4]:
bev_names = {"Cola 1.5L", "CL-150", "Cola 1.5 Ltr", "Orange Juice 1L", "OJ-1L"}
d["product_std"] = d.Product.replace({"CL-150": "Cola 1.5L", "Cola 1.5 Ltr": "Cola 1.5L",
                                       "OJ-1L": "Orange Juice 1L"})
d["cat_std"] = d.Category.str.strip().replace({"Bev.": "Beverages", "": None})
d.loc[d.OrderID == "1008", "amt"] = 360          # decision 4: decimal-point error
dedup = d.drop_duplicates("OrderID", keep="first")  # decision 1: drop duplicate 1004
bev = dedup[dedup.product_std.isin(bev_names) | (dedup.cat_std == "Beverages")]
milk = dedup[dedup.Product.isin(["Fresh Milk 1L", "Milk 1 Litre"])]
strict_total = bev.amt.sum()
print("Beverage rows used:", len(bev))
print(f"STRICT Beverages total (milk excluded): Rs {strict_total:,.0f}")
print(f"Milk rows (blank category): Rs {milk.amt.sum():,.0f}")
print(f"WITH milk as Beverages:                 Rs {strict_total + milk.amt.sum():,.0f}")
naive = raw.Amount.str.replace("Rs", "", regex=False).str.replace(",", "", regex=False).str.strip().astype(float).sum()
print(f"\nNaive SUM of the raw Amount column:     Rs {naive:,.0f}  <- includes the 3600 error, text rows only parse by luck")

Beverage rows used: 13
STRICT Beverages total (milk excluded): Rs 6,320
Milk rows (blank category): Rs 1,320
WITH milk as Beverages:                 Rs 7,640

Naive SUM of the raw Amount column:     Rs 64,010  <- includes the 3600 error, text rows only parse by luck


## Answer

**Total Beverages revenue this morning: Rs 6,320** (Rs 7,640 if the business rules milk as a beverage).

**Confidence: MEDIUM.** Two things would raise it to high: (1) a written business rule for the blank-category milk rows, and (2) confirmation from the branch that order `1008` was really 2 × Rs 180 and not a different pack size.

**One sentence on the dashboard question:** this file cannot be loaded into a dashboard as-is because the dashboard would show six branches instead of three, silently mis-sum the text amounts, and double-count order `1004` — every number on screen would be wrong with no warning.

## The question to answer — preventable at data entry

1. **Branch written four ways** → prevented by a single change: make Branch a **dropdown / lookup** in the POS instead of free text.
2. **Product under three names** → prevented by a single change: **scan the barcode** (or pick from the product master) instead of typing the name.